In [2]:
# Libraries import karo
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = 'notebook_connected'

# Data load karo
df = pd.read_csv(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data\reliance_featured_data.csv',
                 index_col='Date',
                 parse_dates=True)

# Models load karo
models_path = r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models'

with open(f'{models_path}\\lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)

with open(f'{models_path}\\rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

with open(f'{models_path}\\xgb_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print("Data aur Models load ho gaye! ✅")
print("Data Shape:", df.shape)

Data aur Models load ho gaye! ✅
Data Shape: (1335, 30)


In [3]:
# Target variable banao
df['Target'] = df['Close'].shift(-1)
df = df.dropna()

# Features aur Target alag karo
X = df.drop(columns=['Target'])
y = df['Target']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

# Scaling
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# Predictions nikalo teeno models se
lr_pred = scaler_y.inverse_transform(lr_model.predict(X_test_scaled).reshape(-1,1)).ravel()
rf_pred = scaler_y.inverse_transform(rf_model.predict(X_test_scaled).reshape(-1,1)).ravel()
xgb_pred = scaler_y.inverse_transform(xgb_model.predict(X_test_scaled).reshape(-1,1)).ravel()

print("Preprocessing complete! ✅")
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")

Preprocessing complete! ✅
Training rows: 1067
Testing rows:  267


DETAILED ERROR ENALYSIS

In [4]:
# Teeno models ki detailed evaluation
print("=" * 50)
print("DETAILED MODEL EVALUATION REPORT")
print("=" * 50)

models_results = {
    'Linear Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
}

for model_name, pred in models_results.items():
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)
    r2   = r2_score(y_test, pred)
    mape = np.mean(np.abs((y_test - pred) / y_test)) * 100
    
    print(f"\n📊 {model_name}:")
    print(f"   RMSE : ₹{rmse:.2f}")
    print(f"   MAE  : ₹{mae:.2f}")
    print(f"   R²   : {r2:.4f}")
    print(f"   MAPE : {mape:.2f}%")

print("\n" + "=" * 50)

DETAILED MODEL EVALUATION REPORT

📊 Linear Regression:
   RMSE : ₹18.94
   MAE  : ₹14.22
   R²   : 0.9564
   MAPE : 1.02%

📊 Random Forest:
   RMSE : ₹24.86
   MAE  : ₹19.63
   R²   : 0.9248
   MAPE : 1.39%

📊 XGBoost:
   RMSE : ₹28.80
   MAE  : ₹23.08
   R²   : 0.8991
   MAPE : 1.62%



💡 MAPE 1% ka matlab real life mein:

Reliance price ₹1400 hai toh prediction ₹1414 aayegi — sirf ₹14 ka fark!
Ye industry standard se bahut better hai! 😎

In [5]:
# Feature Importance — Random Forest
feature_names = X.columns.tolist()
rf_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Important Features (Random Forest):")
print(rf_importance.head(10))

Top 10 Important Features (Random Forest):
         Feature  Importance
3          Close    0.750852
1           High    0.151372
5           MA_7    0.058140
0           Open    0.011203
2            Low    0.010037
16    Close_Lag1    0.008375
17    Close_Lag2    0.002338
27           OBV    0.000759
22           Day    0.000556
25  Daily_Return    0.000501


💡 Business Insight #12:

Close price 75% important hai! — matlab kal ki price predict karne ke liye aaj ki closing price sabse zyada matter karti hai!
Ye financially bhi sahi hai — traders bhi closing price ko sabse zyada dekhte hain! 😎

In [7]:
# Top 15 features ka chart
fig_imp = go.Figure()

fig_imp.add_trace(go.Bar(
    x=rf_importance['Importance'].head(15),
    y=rf_importance['Feature'].head(15),
    orientation='h',
    marker_color='#7C83FD'
))

fig_imp.update_layout(
    title='Feature Importance — Random Forest (Top 15)',
    xaxis_title='Importance Score',
    yaxis_title='Feature',
    template='plotly_dark',
    height=500,
    yaxis=dict(autorange='reversed')
)

fig_imp.show()

Residual = Actual Price - Predicted Price
Matlab — model kitna galat hua har din!
Agar residuals random hain — model achha hai ✅
Agar residuals mein pattern hai — model mein improvement ki zaroorat hai ⚠️

In [8]:
# Residuals calculate karo
lr_residuals = y_test - lr_pred
rf_residuals = y_test - rf_pred
xgb_residuals = y_test - xgb_pred

# Residual Chart
fig_res = make_subplots(rows=3, cols=1,
                        subplot_titles=(
                            'Linear Regression Residuals',
                            'Random Forest Residuals',
                            'XGBoost Residuals'
                        ))

# Linear Regression
fig_res.add_trace(go.Scatter(
    x=X_test.index,
    y=lr_residuals,
    mode='lines',
    name='LR Residuals',
    line=dict(color='#FF6B6B', width=1)
), row=1, col=1)

# Random Forest
fig_res.add_trace(go.Scatter(
    x=X_test.index,
    y=rf_residuals,
    mode='lines',
    name='RF Residuals',
    line=dict(color='#4ECDC4', width=1)
), row=2, col=1)

# XGBoost
fig_res.add_trace(go.Scatter(
    x=X_test.index,
    y=xgb_residuals,
    mode='lines',
    name='XGB Residuals',
    line=dict(color='#FFE66D', width=1)
), row=3, col=1)

# Zero line teeno mein
for i in range(1, 4):
    fig_res.add_hline(y=0, line_dash="dash",
                      line_color="white",
                      row=i, col=1)

fig_res.update_layout(
    title='Residual Analysis — Teeno Models',
    template='plotly_dark',
    height=700
)

fig_res.show()

# Day 8 - Model Evaluation

## Detailed Results

### Linear Regression 🏆
- RMSE: ₹18.94
- MAE:  ₹14.22
- R²:   0.9564
- MAPE: 1.02%

### Random Forest
- RMSE: ₹24.86
- MAE:  ₹19.63
- R²:   0.9248
- MAPE: 1.39%

### XGBoost
- RMSE: ₹28.80
- MAE:  ₹23.08
- R²:   0.8991
- MAPE: 1.62%

## Feature Importance (Random Forest)
- Close Price: 75% — sabse important!
- High Price: 15%
- MA_7: 5.8%

## Residual Analysis
- Teeno models ke residuals zero ke aaspaas random hain
- Koi systematic bias nahi — models reliable hain